In [ ]:
import random
import numpy as np
import tensorflow as tf

random.seed(0)
np.random.seed(0)
tf.random.set_seed(0)

In [ ]:
!pip install kaggle

import os
import json
from zipfile import ZipFile
from PIL import Image
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [ ]:
# Load kaggle credentials
kaggle_credentials = json.load(open("kaggle.json"))

os.environ['KAGGLE_USERNAME'] = kaggle_credentials["username"]
os.environ['KAGGLE_KEY'] = kaggle_credentials["key"]

In [ ]:
!kaggle datasets download -d abdallahalidev/plantvillage-dataset

Dataset URL: https://www.kaggle.com/datasets/abdallahalidev/plantvillage-dataset
License(s): CC-BY-NC-SA-4.0
 98% 1.99G/2.04G [00:12<00:01, 39.9MB/s]
100% 2.04G/2.04G [00:12<00:00, 175MB/s] 


In [ ]:
with ZipFile("plantvillage-dataset.zip",'r') as zip_ref:
    zip_ref.extractall()

print(os.listdir("plantvillage dataset"))

['segmented', 'color', 'grayscale']


In [ ]:
BASE_DIR = "plantvillage dataset/color"

IMG_SIZE = 224
BATCH_SIZE = 32

In [ ]:
data_gen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2,

    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8,1.2]
)

In [ ]:
train_generator = data_gen.flow_from_directory(
    BASE_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='training',
    class_mode='categorical'
)

validation_generator = data_gen.flow_from_directory(
    BASE_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    subset='validation',
    class_mode='categorical'
)

NUM_CLASSES = train_generator.num_classes
print("Classes:", NUM_CLASSES)

Found 43456 images belonging to 38 classes.
Found 10849 images belonging to 38 classes.
Classes: 38


In [ ]:
base_model = MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze pretrained layers
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       163,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 38)             │         4,902 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,426,854 (9.26 MB)

 Trainable params: 168,870 (659.65 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 744s 534ms/step - accuracy: 0.6590 - loss: 1.2612 - val_accuracy: 0.9049 - val_loss: 0.2994
Epoch 2/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 701s 517ms/step - accuracy: 0.8655 - loss: 0.4242 - val_accuracy: 0.9200 - val_loss: 0.2324
Epoch 3/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 698s 514ms/step - accuracy: 0.8851 - loss: 0.3537 - val_accuracy: 0.9287 - val_loss: 0.2093
Epoch 4/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 699s 514ms/step - accuracy: 0.8951 - loss: 0.3231 - val_accuracy: 0.9277 - val_loss: 0.2079
Epoch 5/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 704s 518ms/step - accuracy: 0.8993 - loss: 0.3107 - val_accuracy: 0.9371 - val_loss: 0.1885
Epoch 6/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 706s 520ms/step - accuracy: 0.9034 - loss: 0.2931 - val_accuracy: 0.9396 - val_loss: 0.1778
Epoch 7/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 683s 503ms/step - accuracy: 0.9091 - loss: 0.2711 - val_accuracy: 0.9415 - val_loss: 0.1749
Epoch 8/10
1358/1358 ━━━━━━━━━━━━━━━━━━━━ 692s 510ms/step - ac

In [ ]:
val_loss, val_acc = model.evaluate(validation_generator)
print("Validation Accuracy:", val_acc*100)

340/340 ━━━━━━━━━━━━━━━━━━━━ 138s 406ms/step - accuracy: 0.9427 - loss: 0.1720
Validation Accuracy: 94.21144723892212


In [ ]:
class_indices = {v:k for k,v in train_generator.class_indices.items()}

json.dump(class_indices, open("class_indices.json","w"))

In [ ]:
def predict_leaf(image_path):

    img = Image.open(image_path).convert("RGB")
    img = img.resize((IMG_SIZE, IMG_SIZE))

    img = np.array(img)/255.0
    img = np.expand_dims(img, axis=0)

    prediction = model.predict(img)[0]

    class_id = np.argmax(prediction)
    disease = class_indices[class_id]
    severity = float(np.max(prediction))

    result = {
        "disease": disease,
        "severity": round(severity,3)
    }

    return result

In [ ]:
print(predict_leaf("tomato_blight.jpg"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step
{'disease': 'Tomato___Late_blight', 'severity': 0.771}


In [ ]:
model.save("plant_disease_model.h5")

In [ ]:
from google.colab import files
files.download("plant_disease_model.h5")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>